In [9]:
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [10]:

# Configurar para mostrar todas las filas sin truncar
pd.set_option('display.max_rows', None)

# (Opcional) Si también quieres ver todas las columnas si fuera necesario
pd.set_option('display.max_columns', None)

In [ ]:

# Set the path to the file you'd like to load
# Para este dataset específico, el archivo se llama 'credit_risk_dataset.csv'
file_path = "loan/loan.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "ranadeep/credit-risk-dataset",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)
print("First 5 records:", df.head())


First 5 records:         id  member_id  loan_amnt  funded_amnt  funded_amnt_inv        term  \
0  1077501    1296599     5000.0       5000.0           4975.0   36 months   
1  1077430    1314167     2500.0       2500.0           2500.0   60 months   
2  1077175    1313524     2400.0       2400.0           2400.0   36 months   
3  1076863    1277178    10000.0      10000.0          10000.0   36 months   
4  1075358    1311748     3000.0       3000.0           3000.0   60 months   

   int_rate  installment grade sub_grade                 emp_title emp_length  \
0     10.65       162.87     B        B2                       NaN  10+ years   
1     15.27        59.83     C        C4                     Ryder   < 1 year   
2     15.96        84.33     C        C5                       NaN  10+ years   
3     13.49       339.31     C        C1       AIR RESOURCES BOARD  10+ years   
4     12.69        67.79     B        B5  University Medical Group     1 year   

  home_ownership  annual_in

In [12]:
df.shape

(887379, 74)

In [ ]:
# 1. Definir el diccionario de mapeo para 0 y 1
mapping = {
    'Fully Paid': 0,
    'Does not meet the credit policy. Status:Fully Paid': 0,
    'Charged Off': 1,
    'Default': 1,
    'Late (31-120 days)': 1,
    'Does not meet the credit policy. Status:Charged Off': 1
}

# 2. Filtrar únicamente los casos válidos (excluyendo los NA)
df = df[df['loan_status'].isin(mapping.keys())].copy()
# 3. Crear la nueva variable objetivo binaria
df['loan_status'] = df['loan_status'].map(mapping)


0     Fully Paid
1    Charged Off
2     Fully Paid
3     Fully Paid
5     Fully Paid
Name: loan_status, dtype: str
0    0
1    1
2    0
3    0
5    0
Name: loan_status, dtype: int64


In [30]:
df.shape

(268385, 40)

In [14]:
columnas_a_excluir = ["url", "last_pymnt_d", "next_pymnt_d", "emp_title", "issue_d", "desc", "title", "zip_code", "addr_state", "earliest_cr_line", "last_credit_pull_d"]
df = df.drop(columns=columnas_a_excluir)

In [15]:
# Muestra el número de valores únicos por cada columna de texto o categórica
df.select_dtypes(include=['object', 'category']).nunique()

/tmp/ipykernel_44931/2292819751.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include=['object', 'category']).nunique()


term                          2
grade                         7
sub_grade                    35
emp_length                   11
home_ownership                6
verification_status           3
pymnt_plan                    2
purpose                      14
initial_list_status           2
application_type              2
verification_status_joint     3
dtype: int64

In [16]:
print("emp_length: " + str(list((df['emp_length'].unique()))))
print("grade: " + str(list((df['grade'].unique()))))
print("sub_grade: " + str(list((df['sub_grade'].unique()))))
print("verification_status: " + str(list((df['verification_status'].unique()))))
print("verification_status_joint: " + str(list((df['verification_status_joint'].unique()))))

emp_length: ['10+ years', '< 1 year', '3 years', '9 years', '4 years', '5 years', '1 year', '6 years', '2 years', '7 years', '8 years', nan]
grade: ['B', 'C', 'A', 'E', 'F', 'D', 'G']
sub_grade: ['B2', 'C4', 'C5', 'C1', 'A4', 'E1', 'F2', 'B5', 'C3', 'B1', 'D1', 'A1', 'B3', 'B4', 'C2', 'D2', 'A3', 'A5', 'D5', 'A2', 'E4', 'D3', 'D4', 'F3', 'E3', 'F1', 'E5', 'G4', 'E2', 'G2', 'G1', 'F5', 'F4', 'G5', 'G3']
verification_status: ['Verified', 'Source Verified', 'Not Verified']
verification_status_joint: [nan, 'Source Verified', 'Not Verified', 'Verified']


In [17]:
# 1. Mapeo para 'grade' (de menor a mayor riesgo, A es mejor que G)
grade_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}
df['grade'] = df['grade'].map(grade_mapping)

# 2. Mapeo automático/ordenado para 'sub_grade' (A1 a G5)
letras = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
subgrade_mapping = {f"{l}{n}": i for i, (l, n) in enumerate(
    (l, n) for l in letras for n in range(1, 6)
)}
df['sub_grade'] = df['sub_grade'].map(subgrade_mapping)

# 3. Mapeo para 'verification_status' (ordenado por nivel de rigor)
verification_mapping = {'Not Verified': 0, 'Source Verified': 1, 'Verified': 2}
df['verification_status'] = df['verification_status'].map(verification_mapping)

# 4. Mapeo para 'verification_status_joint' (manejando los nulos con -1 o NaN)
verification_joint_mapping = {
    'Not Verified': 0, 'Source Verified': 1, 'Verified': 2
}
df['verification_status_joint'] = df['verification_status_joint'].map(
    verification_joint_mapping
)

# Mapeo ordenado para 'emp_length' (los nulos se pueden dejar como -1 o NaN)
emp_length_mapping = {
    '< 1 year': 0,
    '1 year': 1,
    '2 years': 2,
    '3 years': 3,
    '4 years': 4,
    '5 years': 5,
    '6 years': 6,
    '7 years': 7,
    '8 years': 8,
    '9 years': 9,
    '10+ years': 10
}

# Aplicar el mapeo conservando el nombre de la columna
df['emp_length'] = df['emp_length'].map(emp_length_mapping)

In [18]:
# Selección de columnas no numéricas:
columns_to_dummy = df.select_dtypes(exclude=['number'])

# Vista de columnas no numéricas.
columns_to_dummy.head()

,term,home_ownership,pymnt_plan,purpose,initial_list_status,application_type
0,36 months,RENT,n,credit_card,f,INDIVIDUAL
1,60 months,RENT,n,car,f,INDIVIDUAL
2,36 months,RENT,n,small_business,f,INDIVIDUAL
3,36 months,RENT,n,other,f,INDIVIDUAL
5,36 months,RENT,n,wedding,f,INDIVIDUAL


In [19]:
# Función auxiliar (generada con Gemini)
# Codificación con one-hot pero utilizando -1/1 en lugar de 0/1.
def dummy_encode(df, column):
    """
    Encodes a categorical column into dummy variables using -1/1 encoding.

    Args:
        df: The input DataFrame.
        column: The name of the column to encode.

    Returns:
        A DataFrame with the dummy variables.
    """

    dummies = pd.get_dummies(df[column], prefix=column, prefix_sep='_', drop_first=True)
    dummies = dummies.map(lambda x: 1 if x == 1 else -1)  # Convert 0s to -1s
    return pd.concat([df.drop(column, axis=1), dummies], axis=1)

In [20]:
# Aplicación de la función auxiliar para hacer el one-hot encoding:
dummy_encoded = columns_to_dummy.copy()

for column in dummy_encoded.columns:
    dummy_encoded = dummy_encode(dummy_encoded, column)

# Visualización del dataframe recodificado
dummy_encoded.head()

,term_ 60 months,home_ownership_MORTGAGE,home_ownership_NONE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,pymnt_plan_y,purpose_credit_card,purpose_debt_consolidation,purpose_educational,purpose_home_improvement,purpose_house,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding,initial_list_status_w,application_type_JOINT
0,-1,-1,-1,-1,-1,1,-1,1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
1,1,-1,-1,-1,-1,1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
2,-1,-1,-1,-1,-1,1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,1,-1,-1,-1,-1
3,-1,-1,-1,-1,-1,1,-1,-1,-1,-1,-1,-1,-1,-1,-1,1,-1,-1,-1,-1,-1,-1
5,-1,-1,-1,-1,-1,1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,1,-1,-1


In [21]:
# Selección de las columnas numéricas:
numeric_cols = df.drop(columns_to_dummy.columns.tolist(), axis=1)

# Display the first few rows of the selected numeric columns
print(numeric_cols.head())


        id  member_id  loan_amnt  funded_amnt  funded_amnt_inv  int_rate  \
0  1077501    1296599     5000.0       5000.0           4975.0     10.65   
1  1077430    1314167     2500.0       2500.0           2500.0     15.27   
2  1077175    1313524     2400.0       2400.0           2400.0     15.96   
3  1076863    1277178    10000.0      10000.0          10000.0     13.49   
5  1075269    1311441     5000.0       5000.0           5000.0      7.90   

   installment  grade  sub_grade  emp_length  annual_inc  verification_status  \
0       162.87      1          6        10.0     24000.0                    2   
1        59.83      2         13         0.0     30000.0                    1   
2        84.33      2         14        10.0     12252.0                    0   
3       339.31      2         10        10.0     49200.0                    1   
5       156.46      0          3         3.0     36000.0                    1   

   loan_status    dti  delinq_2yrs  inq_last_6mths  mths

In [26]:
# 1. Conteo de nulos por columna
conteo_nulos = df.isnull().sum()

# 2. Porcentaje de nulos respecto al total de filas
porcentaje_nulos = (df.isnull().mean() * 100).round(2)

# 3. Unir ambos resultados en un DataFrame ordenado de mayor a menor porcentaje
df_resumen_nulos = pd.DataFrame({
    'Nulos': conteo_nulos,
    'Porcentaje (%)': porcentaje_nulos
}).sort_values(by='Porcentaje (%)', ascending=False)

print(df_resumen_nulos)


                            Nulos  Porcentaje (%)
emp_length                  10721            3.99
revol_util                    249            0.09
collections_12_mths_ex_med    145            0.05
open_acc                       29            0.01
inq_last_6mths                 29            0.01
total_acc                      29            0.01
pub_rec                        29            0.01
delinq_2yrs                    29            0.01
acc_now_delinq                 29            0.01
int_rate                        0            0.00
term                            0            0.00
funded_amnt_inv                 0            0.00
funded_amnt                     0            0.00
loan_amnt                       0            0.00
member_id                       0            0.00
id                              0            0.00
installment                     0            0.00
pymnt_plan                      0            0.00
verification_status             0            0.00


In [25]:
columnas_con_nulos = [
    'annual_inc_joint',
    'verification_status_joint',
    'dti_joint',
    'il_util',
    'mths_since_rcnt_il',
    'inq_fi',
    'open_rv_24m',
    'open_acc_6m',
    'all_util',
    'inq_last_12m',
    'total_cu_tl',
    'open_il_12m',
    'max_bal_bc',
    'open_il_6m',
    'open_il_24m',
    'open_rv_12m',
    'total_bal_il',
    'desc',
    'mths_since_last_record',
    'mths_since_last_major_derog',
    'mths_since_last_delinq',
    'next_pymnt_d',
    'total_rev_hi_lim',
    'tot_cur_bal',
    'tot_coll_amt'
]

# Eliminar las columnas del DataFrame (axis=1 indica columnas)
df = df.drop(columns=columnas_con_nulos, errors='ignore')

print(f"Columnas restantes en el DataFrame: {df.shape[1]}")

Columnas restantes en el DataFrame: 40


In [29]:
columnas_con_nulos = [
    # 'emp_length',
    # 'last_pymnt_d',
    # 'revol_util',
    # 'title',
    'collections_12_mths_ex_med'
]

df = df.dropna(subset=columnas_con_nulos).copy()
df

KeyboardInterrupt: 

In [ ]:
df.columns

Index(['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv',
       'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length',
       'home_ownership', 'annual_inc', 'verification_status', 'loan_status',
       'pymnt_plan', 'purpose', 'dti', 'delinq_2yrs', 'inq_last_6mths',
       'mths_since_last_delinq', 'mths_since_last_record', 'open_acc',
       'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
       'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt',
       'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int',
       'total_rec_late_fee', 'recoveries', 'collection_recovery_fee',
       'last_pymnt_amnt', 'collections_12_mths_ex_med',
       'mths_since_last_major_derog', 'policy_code', 'application_type',
       'annual_inc_joint', 'dti_joint', 'verification_status_joint',
       'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m',
       'open_il_6m', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il',
       'total_bal_il',